# RealSaS — Arachne Mage A0 FS1 exact execution capsule

Pinned source commit: `32d2202f096eca6613aa965d609d136c40f091b8`  
Historical A0 engine Git blob: `f32d2f75e6b9e6095e1d5455ed9227b32553a9d5`  
FS1 wrapper Git blob: `00ccbd7d79707fa6dab5329baeebac20efb8f470`

Run all cells on a **GPU runtime**. The notebook fails closed: main CUDA cannot run until the exact committed wrapper passes the disposable regression and CPU preflight gates. Disposable checks do **not** count as main scientific A0 optimizer steps.

Private-repo authentication: add `GITHUB_TOKEN` in Colab Secrets, or enter a token when prompted. The token is never printed.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, hashlib, subprocess, getpass, shutil

drive.mount('/content/drive', force_remount=False)

REPO_FULL = 'merynz/RealSaS-OPT'
PINNED_COMMIT = '32d2202f096eca6613aa965d609d136c40f091b8'
DRIVE_ROOT = Path('/content/drive/MyDrive/RealSaS_ARACHNE_MAGE_A0_FS1_20260908')
REPO_DIR = Path('/content/realsas_opt')
CHECK_DIR = Path('/content/arachne_a0_fs1_exact_checks')
MAIN_OUT = DRIVE_ROOT / 'A0_MAIN_FS1'

EXPECTED_HIST_BLOB = 'f32d2f75e6b9e6095e1d5455ed9227b32553a9d5'
EXPECTED_WRAPPER_BLOB = '00ccbd7d79707fa6dab5329baeebac20efb8f470'
EXPECTED_CACHE_SHA = 'db87c42d65e777072b3a607178a2c7f19ab221a4969c380eac46070db2216edd'
EXPECTED_CACHE_MANIFEST_SHA = '5b3e63d60fce76226e16aac5d754a5f9cde114ccd8327a3242fad8a714af7d6b'
EXPECTED_ZERO_SHA = '987f7d18ce202454c4ea5101225bfaed54aeb4638cba1077e70efc15f2038e9b'
EXPECTED_SKELETON_FILE_SHA = '48754ad703c596ec9d332c6f733f1dd31e74d016ef15f3ce451263a724493992'
EXPECTED_BINDING_SHA = 'ab74756e32ee5c9f4f2d4020cdb56620a110130d80d7b9384c62509af3f193cf'
EXPECTED_CACHE_BINDING_SHA = 'c7e3bf10fc8edf16f862b4ce58b3aabfeb2aabf14cc8e744e53e3afc0764ac9f'
EXPECTED_SUPERVISED = 934
EXPECTED_LOW = 16
CAMERA_SHA = ['73004e0654b576e0c51893af544e0af8fcc4e613ce07ea9884272285d55cd541', 'bdc172a4aff332f956d1403e36b2f8684b68059fdc82f9efddf35d05a6d9b4d4', '3c2bbc44ef9005b4a545a3381205a5d6a92af15b4791c9075071b8cad02a1a6c', '24b2f115d908422d885f85e956fcc36ac78fd0c90b503f698caa62febc2b9c4d', '5bf00783d6509c2ca142e05ef705d5cdb5df17ad248b782d2fe8cf8a297bee39', '7ee3e50739318eeb122b5b0ec67260dd32e21d949398f48c408a6c239e5c89fe', 'daa19fa58ff602977d64b720c4198956809855149d814df487c7762a963f1eec', '68f51fbfce4c31f94281e1569d74b44609435285668f8a8b1b278e76db6ea53f']

CACHE = DRIVE_ROOT / 'ARACHNE_MAGE_FS1_CONDITIONING_CACHE_V2.npz'
CACHE_MANIFEST = DRIVE_ROOT / 'ARACHNE_MAGE_FS1_CONDITIONING_CACHE_V2_MANIFEST.json'
ZERO = DRIVE_ROOT / 'ZERO_SURFACE_PRODUCT_CLIPPED.npz'
SKELETON = DRIVE_ROOT / 'FINAL_QUALIFIED_SKELETON_IR.json'

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8<<20), b''): h.update(b)
    return h.hexdigest()

def git_blob_sha(p):
    b=Path(p).read_bytes()
    return hashlib.sha1(f'blob {len(b)}\0'.encode('ascii')+b).hexdigest()

def run(cmd, *, env=None, cwd=None):
    print('+', ' '.join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], check=True, env=env, cwd=cwd)

try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if not token:
    token = getpass.getpass('GitHub token (repo read access; hidden): ').strip()
if not token:
    raise RuntimeError('GITHUB_TOKEN_REQUIRED_FOR_PRIVATE_EXACT_CLONE')

if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
env_git=os.environ.copy(); env_git['GIT_TERMINAL_PROMPT']='0'
askpass=Path('/content/.realsas_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1].lower() if len(sys.argv)>1 else ''\nprint('x-access-token' if 'username' in p else os.environ['REALSAS_GITHUB_TOKEN'])\n",encoding='utf-8')
askpass.chmod(0o700)
env_git['GIT_ASKPASS']=str(askpass); env_git['REALSAS_GITHUB_TOKEN']=token
clone_url=f'https://github.com/{REPO_FULL}.git'
try:
    subprocess.run(['git','clone','--quiet','--no-checkout',clone_url,str(REPO_DIR)],check=True,env=env_git)
finally:
    env_git.pop('REALSAS_GITHUB_TOKEN',None)
    token=None
    try: askpass.unlink()
    except FileNotFoundError: pass
run(['git','checkout','--quiet',PINNED_COMMIT],cwd=REPO_DIR)
head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip()
if head != PINNED_COMMIT: raise RuntimeError(f'PINNED_COMMIT_DRIFT:{head}')
print('PINNED_COMMIT_OK', head)


In [ ]:
HIST = REPO_DIR / 'experiments/arachne_skintokens_fit1_20260908/run_arachne_mage_a0_historical_1cc449.py'
RUNNER = REPO_DIR / 'experiments/arachne_skintokens_fit1_20260908/run_arachne_mage_a0_fs1_v2.py'
if git_blob_sha(HIST) != EXPECTED_HIST_BLOB: raise RuntimeError('HISTORICAL_ENGINE_GIT_BLOB_DRIFT')
if git_blob_sha(RUNNER) != EXPECTED_WRAPPER_BLOB: raise RuntimeError('FS1_WRAPPER_GIT_BLOB_DRIFT')

required={CACHE:EXPECTED_CACHE_SHA,CACHE_MANIFEST:EXPECTED_CACHE_MANIFEST_SHA,ZERO:EXPECTED_ZERO_SHA,SKELETON:EXPECTED_SKELETON_FILE_SHA}
for p,exp in required.items():
    if not p.is_file(): raise FileNotFoundError(p)
    got=sha256_file(p)
    if got!=exp: raise RuntimeError(f'INPUT_SHA_DRIFT:{p.name}:{got}:{exp}')
for i,exp in enumerate(CAMERA_SHA):
    p=DRIVE_ROOT/f'V{i}.camera.json'
    if not p.is_file(): raise FileNotFoundError(p)
    got=sha256_file(p)
    if got!=exp: raise RuntimeError(f'CAMERA_SHA_DRIFT_V{i}:{got}:{exp}')

m=json.loads(CACHE_MANIFEST.read_text())
if m['cache']['binding_sha256']!=EXPECTED_CACHE_BINDING_SHA: raise RuntimeError('CACHE_BINDING_DRIFT')
if m['inputs']['fs1_target_binding_sha256']!=EXPECTED_BINDING_SHA: raise RuntimeError('TARGET_BINDING_DRIFT')
if m['cache']['supervised_rows']!=EXPECTED_SUPERVISED or m['cache']['low_rows']!=EXPECTED_LOW: raise RuntimeError('SUPERVISION_DRIFT')
print('EXACT_SOURCE_AND_INPUT_AUTHORITY_PASS')


In [ ]:
if CHECK_DIR.exists(): shutil.rmtree(CHECK_DIR)
REG_DIR=CHECK_DIR/'regression'; REG_DIR.mkdir(parents=True)
base=[sys.executable,str(RUNNER),'--cache',str(CACHE),'--cache-manifest',str(CACHE_MANIFEST),'--zero-surface',str(ZERO),'--camera-dir',str(DRIVE_ROOT),'--qualified-skeleton',str(SKELETON)]
env_cpu=os.environ.copy(); env_cpu['PYTHONPATH']=str(REPO_DIR); env_cpu['CUDA_VISIBLE_DEVICES']=''
run(base+['--output-dir',str(REG_DIR),'--regression-only'],env=env_cpu,cwd=REPO_DIR)
r=json.loads((REG_DIR/'ARACHNE_MAGE_A0_FS1_RUNNER_REGRESSION.json').read_text())
if r.get('status')!='PASS_CPU_RESUME_AND_TERMINAL_IDEMPOTENCE_REGRESSIONS': raise RuntimeError('EXACT_REGRESSION_NOT_PASS')
for k in ['exact_resume','model_state_exact','optimizer_state_exact','scheduler_state_exact','rng_state_exact','loss_trace_exact','lr_exact','terminal_no_closure_idempotent','terminal_pass_idempotent']:
    if r.get(k) is not True: raise RuntimeError(f'EXACT_REGRESSION_GATE_FAIL:{k}')
if r.get('main_scientific_a0_optimizer_steps')!=0: raise RuntimeError('MAIN_A0_STEP_GUARD_FAIL')
print('EXACT_COMMITTED_WRAPPER_REGRESSION_PASS')


In [ ]:
PRE_DIR=CHECK_DIR/'preflight'; PRE_DIR.mkdir(parents=True)
run(base+['--output-dir',str(PRE_DIR),'--preflight-only'],env=env_cpu,cwd=REPO_DIR)
p=json.loads((PRE_DIR/'ARACHNE_MAGE_A0_PREFLIGHT.json').read_text())
if p.get('status')!='PASS_CPU_EXECUTABLE_PREFLIGHT': raise RuntimeError('EXACT_CPU_PREFLIGHT_NOT_PASS')
if p.get('supervised_rows')!=EXPECTED_SUPERVISED: raise RuntimeError('PREFLIGHT_SUPERVISION_DRIFT')

seal={
 'schema':'RealSaS.ArachneMageA0FS1ExactClonePackageSeal.v1',
 'status':'PASS_EXACT_COMMITTED_WRAPPER_CPU_REGRESSION_AND_PREFLIGHT__MAIN_CUDA_AUTHORIZED_BY_PACKAGE_GATE',
 'pinned_commit':PINNED_COMMIT,
 'historical_engine_git_blob_sha':git_blob_sha(HIST),
 'wrapper_git_blob_sha':git_blob_sha(RUNNER),
 'cache_sha256':sha256_file(CACHE),
 'cache_manifest_sha256':sha256_file(CACHE_MANIFEST),
 'target_binding_sha256':EXPECTED_BINDING_SHA,
 'cache_binding_sha256':EXPECTED_CACHE_BINDING_SHA,
 'supervised_rows':EXPECTED_SUPERVISED,
 'low_rows':EXPECTED_LOW,
 'regression_report_sha256':sha256_file(REG_DIR/'ARACHNE_MAGE_A0_FS1_RUNNER_REGRESSION.json'),
 'preflight_report_sha256':sha256_file(PRE_DIR/'ARACHNE_MAGE_A0_PREFLIGHT.json'),
 'main_scientific_a0_optimizer_steps_before_main':0,
 'a1_optimizer_authorized':False,
}
SEAL=DRIVE_ROOT/'ARACHNE_MAGE_A0_FS1_EXACT_CLONE_PACKAGE_SEAL.json'
SEAL.write_text(json.dumps(seal,indent=2,sort_keys=True)+'\n')
print('EXACT_PACKAGE_SEAL', SEAL, sha256_file(SEAL))
print(json.dumps(seal,indent=2,sort_keys=True))


In [ ]:
# Main scientific A0. This cell is fail-closed on the exact package seal above.
RUN_MAIN_A0_CUDA = True
if RUN_MAIN_A0_CUDA:
    seal=json.loads((DRIVE_ROOT/'ARACHNE_MAGE_A0_FS1_EXACT_CLONE_PACKAGE_SEAL.json').read_text())
    if seal.get('status')!='PASS_EXACT_COMMITTED_WRAPPER_CPU_REGRESSION_AND_PREFLIGHT__MAIN_CUDA_AUTHORIZED_BY_PACKAGE_GATE': raise RuntimeError('EXACT_PACKAGE_SEAL_REQUIRED')
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('CUDA_RUNTIME_REQUIRED_FOR_MAIN_A0')
    MAIN_OUT.mkdir(parents=True,exist_ok=True)
    env_main=os.environ.copy(); env_main['PYTHONPATH']=str(REPO_DIR); env_main.pop('CUDA_VISIBLE_DEVICES',None)
    run(base+['--output-dir',str(MAIN_OUT),'--require-cuda'],env=env_main,cwd=REPO_DIR)
    result_path=MAIN_OUT/'ARACHNE_MAGE_A0_RESULT.json'
    if not result_path.is_file(): raise RuntimeError('MAIN_A0_RESULT_MISSING')
    result=json.loads(result_path.read_text())
    print('MAIN_A0_STATUS', result.get('status'), 'final_step', result.get('final_step'), 'terminal_streak', result.get('terminal_streak'))
    print(json.dumps({k:v for k,v in result.items() if k!='trace'},indent=2,sort_keys=True,default=float))
else:
    print('Main CUDA execution disabled by RUN_MAIN_A0_CUDA=False')
